In [1]:
"""
Cross-Project Vulnerability Detection (GAN Version)
---------------------------------------------------
Train on Debian → Test on Chrome
Handles class imbalance using GAN
"""

# =========================================================
# Imports
# =========================================================

import os
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    roc_auc_score,
    f1_score,
    confusion_matrix
)

from xgboost import XGBClassifier

from keras.models import Sequential, Model
from keras.layers import Dense, Dropout, LeakyReLU, Input
from keras.optimizers import Adam
from keras.callbacks import Callback

warnings.filterwarnings("ignore")

# =========================================================
# TensorFlow Configuration
# =========================================================

os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

tf.config.run_functions_eagerly(True)

tf.config.threading.set_inter_op_parallelism_threads(0)
tf.config.threading.set_intra_op_parallelism_threads(0)

os.environ["OMP_NUM_THREADS"] = "8"

# =========================================================
# Progress Logger
# =========================================================

class BatchLogger(Callback):

    def on_train_batch_begin(self, batch, logs=None):
        if batch % 50 == 0:
            print(f"Processing batch {batch}")

# =========================================================
# Reproducibility
# =========================================================

seed = 42

np.random.seed(seed)
tf.random.set_seed(seed)
random.seed(seed)

os.environ["PYTHONHASHSEED"] = str(seed)

# =========================================================
# Original Configuration
# =========================================================

EPOCHS_INITIAL = 50
EPOCHS_RETRAIN = 30

BATCH_SIZE_INITIAL = 64
BATCH_SIZE_RETRAIN = 32

LEARNING_RATE = 1e-5
NUM_ITERATIONS = 3

# =========================================================
# Neural Network
# =========================================================

def create_neural_network(input_dim: int):

    model = Sequential([
        Dense(256, activation="relu", input_dim=input_dim),
        Dropout(0.2),

        Dense(128, activation="relu"),
        Dropout(0.2),

        Dense(64, activation="relu"),
        Dropout(0.2),

        Dense(32, activation="relu"),
        Dropout(0.2),

        Dense(16, activation="relu"),

        Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


# =========================================================
# GAN for Minority Class Generation
# =========================================================

def build_generator(latent_dim, output_dim):

    model = Sequential([
        Dense(128, input_dim=latent_dim),
        LeakyReLU(alpha=0.2),

        Dense(256),
        LeakyReLU(alpha=0.2),

        Dense(output_dim, activation="tanh")
    ])

    return model


def build_discriminator(input_dim):

    model = Sequential([
        Dense(256, input_dim=input_dim),
        LeakyReLU(alpha=0.2),
        Dropout(0.3),

        Dense(128),
        LeakyReLU(alpha=0.2),
        Dropout(0.3),

        Dense(1, activation="sigmoid")
    ])

    model.compile(
        loss="binary_crossentropy",
        optimizer=Adam(0.0002),
        metrics=["accuracy"]
    )

    return model


def train_gan(minority_data, latent_dim=100, epochs=3000, batch_size=64):

    input_dim = minority_data.shape[1]

    generator = build_generator(latent_dim, input_dim)
    discriminator = build_discriminator(input_dim)

    discriminator.trainable = False

    noise = Input(shape=(latent_dim,))
    generated = generator(noise)
    validity = discriminator(generated)

    gan = Model(noise, validity)

    gan.compile(loss="binary_crossentropy", optimizer=Adam(0.0002))

    half_batch = batch_size // 2

    for epoch in range(epochs):

        idx = np.random.randint(0, minority_data.shape[0], half_batch)
        real_samples = minority_data[idx]

        noise = np.random.normal(0, 1, (half_batch, latent_dim))
        fake_samples = generator.predict(noise, verbose=0)

        discriminator.trainable = True
        discriminator.train_on_batch(real_samples, np.ones((half_batch, 1)))
        discriminator.train_on_batch(fake_samples, np.zeros((half_batch, 1)))
        discriminator.trainable = False

        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        gan.train_on_batch(noise, np.ones((batch_size, 1)))

        if epoch % 500 == 0:
            print(f"GAN Training Epoch {epoch}/{epochs}")

    return generator


def generate_synthetic_samples(generator, n_samples, latent_dim=100):

    noise = np.random.normal(0, 1, (n_samples, latent_dim))
    synthetic = generator.predict(noise, verbose=0)

    return synthetic


# =========================================================
# Semi-supervised Transfer Learning
# =========================================================

def semi_supervised_transfer_learning(
        x_train,
        y_train,
        x_test):

    model = create_neural_network(x_train.shape[1])

    print("\nTraining Neural Network on Debian (GAN balanced)...")

    model.fit(
        x_train,
        y_train,
        epochs=EPOCHS_INITIAL,
        batch_size=BATCH_SIZE_INITIAL,
        validation_split=0.2,
        callbacks=[BatchLogger()],
        verbose=1
    )

    print("\nGenerating pseudo labels for Chrome dataset...")

    y_pred = model.predict(x_test, verbose=0)

    y_pred_binary = (y_pred > 0.5).astype(int).flatten()

    x_train_aug = np.concatenate((x_train, x_test))
    y_train_aug = np.concatenate((y_train, y_pred_binary))

    print("\nRetraining model with pseudo-labelled Chrome data...")

    model.fit(
        x_train_aug,
        y_train_aug,
        epochs=EPOCHS_RETRAIN,
        batch_size=BATCH_SIZE_RETRAIN,
        verbose=1
    )

    return model, y_pred.flatten()


# =========================================================
# XGBoost
# =========================================================

def train_base_model_xgb(x_train, y_train):

    model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        use_label_encoder=False,
        eval_metric="logloss"
    )

    model.fit(x_train, y_train)

    return model


# =========================================================
# Main
# =========================================================

def main():

    print("\n=== Training on Debian → Testing on Chrome (GAN) ===")

    # -----------------------------------------------------
    # Load Embeddings
    # -----------------------------------------------------

    deb_emb = np.load("dataset/debian_embeddings.npy")
    deb_lbl = np.load("dataset/debian_labels.npy")

    chr_emb = np.load("dataset/chrome_embeddings.npy")
    chr_lbl = np.load("dataset/chrome_labels.npy")

    deb_emb = deb_emb.astype(np.float32)
    chr_emb = chr_emb.astype(np.float32)

    deb_lbl = deb_lbl.astype(np.float32)
    chr_lbl = chr_lbl.astype(np.float32)

    print("Debian dataset shape:", deb_emb.shape)
    print("Chrome dataset shape:", chr_emb.shape)

    # -----------------------------------------------------
    # Normalize
    # -----------------------------------------------------

    scaler = StandardScaler()

    deb_emb = scaler.fit_transform(deb_emb)
    chr_emb = scaler.transform(chr_emb)

    # -----------------------------------------------------
    # GAN Balancing
    # -----------------------------------------------------

    print("\nApplying GAN to balance Debian dataset...")

    minority_data = deb_emb[deb_lbl == 1]
    majority_data = deb_emb[deb_lbl == 0]

    n_majority = len(majority_data)
    n_minority = len(minority_data)

    print("Minority samples:", n_minority)
    print("Majority samples:", n_majority)

    generator = train_gan(minority_data)

    n_generate = n_majority - n_minority

    synthetic_samples = generate_synthetic_samples(generator, n_generate)

    synthetic_labels = np.ones(n_generate)

    deb_emb_gan = np.vstack((deb_emb, synthetic_samples))
    deb_lbl_gan = np.concatenate((deb_lbl, synthetic_labels))

    print("After GAN balancing:", deb_emb_gan.shape)

    # -----------------------------------------------------
    # Ensemble Training
    # -----------------------------------------------------

    ensemble_predictions = np.zeros(len(chr_lbl))

    for i in range(NUM_ITERATIONS):

        print(f"\n=== Ensemble Iteration {i+1}/{NUM_ITERATIONS} ===")

        model_nn, _ = semi_supervised_transfer_learning(
            deb_emb_gan,
            deb_lbl_gan,
            chr_emb
        )

        print("\nTraining XGBoost model...")

        model_xgb = train_base_model_xgb(
            deb_emb_gan,
            deb_lbl_gan
        )

        print("Predicting Chrome dataset...")

        model_xgb_pred = model_xgb.predict(chr_emb)

        ensemble_predictions += model_xgb_pred

    # -----------------------------------------------------
    # Average Predictions
    # -----------------------------------------------------

    ensemble_avg = ensemble_predictions / NUM_ITERATIONS

    y_pred_final = (ensemble_avg > 0.5).astype(int)

    # -----------------------------------------------------
    # Metrics
    # -----------------------------------------------------

    accuracy = accuracy_score(chr_lbl, y_pred_final)
    recall = recall_score(chr_lbl, y_pred_final)
    precision = precision_score(chr_lbl, y_pred_final)
    auc = roc_auc_score(chr_lbl, ensemble_avg)
    f1 = f1_score(chr_lbl, y_pred_final)

    tn, fp, fn, tp = confusion_matrix(chr_lbl, y_pred_final).ravel()

    g_mean = np.sqrt(tp / (tp + fn))
    pf = fp / (fp + tn)

    print("\n=== Evaluation Results (Chrome Test Set) ===")

    print(f"Accuracy:  {accuracy:.3f}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall:    {recall:.3f}")
    print(f"F1-score:  {f1:.3f}")
    print(f"AUC:       {auc:.3f}")
    print(f"G-mean:    {g_mean:.3f}")
    print(f"PF value:  {pf:.3f}")

    print("\nConfusion Matrix:")
    print(f"TN: {tn}")
    print(f"FP: {fp}")
    print(f"FN: {fn}")
    print(f"TP: {tp}")


# =========================================================

if __name__ == "__main__":
    main()


=== Training on Debian → Testing on Chrome (GAN) ===
Debian dataset shape: (18298, 1024)
Chrome dataset shape: (4436, 1024)

Applying GAN to balance Debian dataset...
Minority samples: 1415
Majority samples: 16883
GAN Training Epoch 0/3000
GAN Training Epoch 500/3000
GAN Training Epoch 1000/3000
GAN Training Epoch 1500/3000
GAN Training Epoch 2000/3000
GAN Training Epoch 2500/3000
After GAN balancing: (33766, 1024)

=== Ensemble Iteration 1/3 ===

Training Neural Network on Debian (GAN balanced)...
Epoch 1/50
Processing batch 0
 49/423 ━━━━━━━━━━━━━━━━━━━━ 6s 17ms/step - accuracy: 0.4319 - loss: 0.8359Processing batch 50
100/423 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.4458 - loss: 0.8133Processing batch 100
148/423 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 0.4588 - loss: 0.7967Processing batch 150
198/423 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.4720 - loss: 0.7823Processing batch 200
249/423 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.4843 - loss: 0.7700Process